# CS 181 Homework 5

This notebook keeps the original assignment scaffold and is organized around the four homework problems:

- **Problem 1: SimCLR**
- **Problem 2: GANs**
- **Problem 3: Clustering**
- **Problem 4: PCA**

## How to read this notebook

1. Start each problem by reading the overview-and-roadmap markdown cell.
2. Then read the code cells in order; they follow the intended implementation flow from the assignment.
3. Generated clustering and PCA figures are saved in `img_output/`.
4. Some training cells are intentionally expensive; read the notes before running them end to end.


## Problem 1: SimCLR


### Overview and Roadmap

This section is meant to be read in the same order a student would work the problem. The notebook structure mirrors the assignment scaffold rather than jumping straight to the final training loop.

#### What this problem asks you to do

1. Understand the encoder architecture that produces feature vectors.
2. Implement the projection head used only during contrastive training.
3. Implement the NT-Xent objective carefully and verify it on a hand-worked example.
4. Use the provided augmentation pipeline to pre-train without labels.
5. Freeze the encoder, train a linear classifier on top, and compare against supervised training.

#### How the code below is organized

- The early code cells define reusable building blocks.
- The middle cells check correctness before expensive training runs.
- The later cells run the full SimCLR pipeline and summarize the comparison.


The imports are grouped to support the full SimCLR pipeline: PyTorch for the model and optimization, `torchvision` for FashionMNIST and augmentations, and `matplotlib` for quick visual checks.


### Setup and Imports


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

### Section 1.1: Backbone Encoder


In [ ]:
BASE_CHANNELS = 32
NUM_BLOCKS = 3


class ResBlock(nn.Module):
    """Basic residual block: two 3x3 convs with BN and a skip connection."""

    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x):
        identity = x
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + identity)


class SimCLREncoder(nn.Module):
    """ResNet encoder for SimCLR (no classification head)."""

    def __init__(self, num_blocks=NUM_BLOCKS, base_channels=BASE_CHANNELS):
        super().__init__()
        C = base_channels
        self.out_dim = C

        self.stem = nn.Sequential(
            nn.Conv2d(1, C, 3, padding=1, bias=False),
            nn.BatchNorm2d(C),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.blocks = nn.Sequential(*[ResBlock(C) for _ in range(num_blocks)])
        self.pool = nn.AdaptiveAvgPool2d(1)

    def forward(self, x):
        x = self.stem(x)
        x = self.blocks(x)
        x = self.pool(x)
        return torch.flatten(x, 1)  # (B, C)


enc_test = SimCLREncoder()
dummy = torch.randn(2, 1, 28, 28)
print(f"Encoder output shape: {enc_test(dummy).shape}")  # (2, 32)
print(f"Encoder parameters: {sum(p.numel() for p in enc_test.parameters()):,}")

### Section 1.2: Implementing SimCLR Components


#### Subpart 1.2.a: Projection Head


In [ ]:
class ProjectionHead(nn.Module):
    """Two-layer MLP projection head for SimCLR."""

    def __init__(self, input_dim, hidden_dim=128, output_dim=64):
        super().__init__()
        # TODO: Define a 2-layer MLP: Linear -> ReLU -> Linear
        # sol
        # YOUR CODE HERE
        pass
        # los

    def forward(self, h):
        # TODO: Pass h through the MLP and return the output.
        # sol
        # YOUR CODE HERE
        pass
        # los


#### Subpart 1.2.b: NT-Xent Loss


In [ ]:
def nt_xent_loss(z1, z2, temperature=0.5):
    """
    NT-Xent (Normalized Temperature-scaled Cross-Entropy) loss.

    Args:
        z1: (N, D) projected representations of first augmented views
        z2: (N, D) projected representations of second augmented views
        temperature: temperature scaling parameter tau

    Returns:
        Scalar loss (mean over all 2N positive pairs)
    """
    N = z1.shape[0]

    # TODO: Implement the NT-Xent loss.
    # 1. L2-normalize z1 and z2.
    # 2. Concatenate into a (2N, D) matrix.
    # 3. Compute the (2N, 2N) cosine similarity matrix, scaled by 1/temperature.
    # 4. Mask out the diagonal (self-similarity) by setting it to -inf.
    # 5. Create labels: for i in [0, N), the positive pair is at index i+N;
    #    for i in [N, 2N), the positive pair is at index i-N.
    # 6. Compute and return the cross-entropy loss.

    # sol
    # YOUR CODE HERE
    pass
    # los

    return loss

#### Check 1.2.c: Hand Calculation


In [ ]:
z1_test = torch.tensor([[1.0, 0.0], [0.0, 2.0]])   # views 1,2 (images A, B)
z2_test = torch.tensor([[3.0, 4.0], [-4.0, 3.0]])   # views 3,4 (images A, B)

loss_hand = nt_xent_loss(z1_test, z2_test, temperature=1.0)
print(f"NT-Xent loss (tau=1.0): {loss_hand.item():.4f}")
assert abs(loss_hand.item() - 0.802) < 0.01, f"Loss mismatch: {loss_hand.item()}"

#### Check 1.2.d: Public Test Cases


In [ ]:
def test_nt_xent_hand_calculation():
    """Public test: matches the hand-calculation check above."""
    z1 = torch.tensor([[1.0, 0.0], [0.0, 2.0]])
    z2 = torch.tensor([[3.0, 4.0], [-4.0, 3.0]])
    loss = nt_xent_loss(z1, z2, temperature=1.0)
    assert abs(loss.item() - 0.802) < 0.01
    print("  [PASS] test_nt_xent_hand_calculation")


def test_nt_xent_temperature_effect():
    """Lower temperature should give higher loss (sharper distribution)."""
    z1 = torch.randn(16, 8)
    z2 = torch.randn(16, 8)
    loss_low_t = nt_xent_loss(z1, z2, temperature=0.1)
    loss_high_t = nt_xent_loss(z1, z2, temperature=2.0)
    assert loss_low_t.item() > loss_high_t.item()
    print("  [PASS] test_nt_xent_temperature_effect")


# Run the public tests
test_nt_xent_hand_calculation()
test_nt_xent_temperature_effect()
print("\nPublic tests passed!")


### Section 1.3: SimCLR Training and Linear Evaluation


#### Subpart 1.3.a: Data Loading and Augmentations


In [ ]:
class SimCLRAugmentation:
    """Returns two different augmented views of the same image."""

    def __init__(self):
        self.transform = transforms.Compose([
            transforms.RandomResizedCrop(28, scale=(0.6, 1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(15),
            transforms.RandomApply(
                [transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0))],
                p=0.5,
            ),
            transforms.ToTensor(),
        ])

    def __call__(self, x):
        return self.transform(x), self.transform(x)


simclr_aug = SimCLRAugmentation()

train_ds_simclr = torchvision.datasets.FashionMNIST(
    root="./data", train=True, download=True, transform=simclr_aug,
)
train_ds_supervised = torchvision.datasets.FashionMNIST(
    root="./data", train=True, download=True, transform=transforms.ToTensor(),
)
test_ds = torchvision.datasets.FashionMNIST(
    root="./data", train=False, download=True, transform=transforms.ToTensor(),
)

print(f"Training set size: {len(train_ds_simclr)}")
print(f"Test set size: {len(test_ds)}")

##### Visual Check: Augmentations


In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(6, 6))
for row in range(3):
    img, _ = train_ds_supervised[row]
    axes[row, 0].imshow(img.squeeze(), cmap="gray")
    axes[row, 0].set_title("Original" if row == 0 else "")

    (v1, v2), _ = train_ds_simclr[row]
    axes[row, 1].imshow(v1.squeeze(), cmap="gray")
    axes[row, 1].set_title("View 1" if row == 0 else "")
    axes[row, 2].imshow(v2.squeeze(), cmap="gray")
    axes[row, 2].set_title("View 2" if row == 0 else "")

for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.show()

#### Subpart 1.3.b: SimCLR Pre-training


In [ ]:
# Hyperparameters
SIMCLR_EPOCHS = 10
SIMCLR_BATCH_SIZE = 512
SIMCLR_LR = 1e-3
SIMCLR_TEMPERATURE = 0.5
PROJECTION_HIDDEN = 128
PROJECTION_DIM = 64

RUN_SIMCLR_TRAINING = True  # Set to False if you want to skip the expensive training run while developing earlier parts

In [ ]:
def train_simclr(encoder, proj_head, train_loader, epochs, lr, temperature, device):
    """
    Train SimCLR: encoder + projection head with NT-Xent loss.

    Args:
        encoder: SimCLREncoder instance
        proj_head: ProjectionHead instance
        train_loader: DataLoader yielding ((view1, view2), labels)
        epochs: number of training epochs
        lr: learning rate
        temperature: NT-Xent temperature
        device: torch device

    Returns:
        List of average loss per epoch
    """
    params = list(encoder.parameters()) + list(proj_head.parameters())
    optimizer = optim.Adam(params, lr=lr)
    loss_history = []

    encoder.train()
    proj_head.train()

    for epoch in range(epochs):
        total_loss = 0.0
        num_batches = 0

        for (x1, x2), _ in train_loader:
            # TODO: Implement the SimCLR training step.
            # 1. Move x1, x2 to device.
            # 2. Pass both views through the encoder to get h1, h2.
            # 3. Pass h1, h2 through the projection head to get z1, z2.
            # 4. Compute the NT-Xent loss.
            # 5. Backpropagate and update weights.

            # sol
            # YOUR CODE HERE
            pass
            # los

            total_loss += loss.item()
            num_batches += 1

        avg_loss = total_loss / num_batches
        loss_history.append(avg_loss)
        print(f"  Epoch {epoch+1:2d}/{epochs}  loss={avg_loss:.4f}")

    return loss_history

In [ ]:
if RUN_SIMCLR_TRAINING:
    torch.manual_seed(SEED)

    simclr_encoder = SimCLREncoder(NUM_BLOCKS, BASE_CHANNELS).to(device)
    simclr_proj = ProjectionHead(
        input_dim=BASE_CHANNELS,
        hidden_dim=PROJECTION_HIDDEN,
        output_dim=PROJECTION_DIM,
    ).to(device)

    simclr_loader = DataLoader(
        train_ds_simclr, batch_size=SIMCLR_BATCH_SIZE, shuffle=True,
        num_workers=0, pin_memory=True, drop_last=True,
    )

    print("SimCLR pre-training, it may takes ~20 minutes on CPU:")
    simclr_losses = train_simclr(
        simclr_encoder, simclr_proj, simclr_loader,
        epochs=SIMCLR_EPOCHS, lr=SIMCLR_LR,
        temperature=SIMCLR_TEMPERATURE, device=device,
    )

    plt.figure(figsize=(8, 4))
    plt.plot(range(1, len(simclr_losses) + 1), simclr_losses, marker="o")
    plt.xlabel("Epoch")
    plt.ylabel("NT-Xent Loss")
    plt.title("SimCLR Pre-training Loss")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

#### Subpart 1.3.c: Linear Evaluation


In [ ]:
@torch.no_grad()
def extract_representations(encoder, dataset, device, batch_size=512):
    """Extract encoder representations for an entire dataset."""
    encoder.eval()
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    all_h, all_y = [], []
    for x, y in loader:
        h = encoder(x.to(device))
        all_h.append(h.cpu())
        all_y.append(y)
    return torch.cat(all_h), torch.cat(all_y)


def linear_eval(encoder, train_dataset, test_dataset, device,
                epochs=20, lr=1e-2, batch_size=256):
    """
    Linear evaluation protocol: train a linear classifier on frozen representations.

    Args:
        encoder: pre-trained encoder (will be frozen)
        train_dataset: training dataset (with ToTensor transform)
        test_dataset: test dataset
        device: torch device
        epochs: number of training epochs for the linear classifier
        lr: learning rate
        batch_size: batch size

    Returns:
        test_accuracy: float
    """
    print("  Extracting representations...")
    h_train, y_train = extract_representations(encoder, train_dataset, device)
    h_test, y_test = extract_representations(encoder, test_dataset, device)

    train_loader = DataLoader(
        TensorDataset(h_train, y_train),
        batch_size=batch_size, shuffle=True,
    )

    classifier = nn.Linear(encoder.out_dim, 10).to(device)
    optimizer = optim.Adam(classifier.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    classifier.train()
    for epoch in range(epochs):
        total_loss = 0.0
        for h_batch, y_batch in train_loader:
            h_batch, y_batch = h_batch.to(device), y_batch.to(device)
            logits = classifier(h_batch)
            loss = criterion(logits, y_batch)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        if (epoch + 1) % 5 == 0:
            print(f"    Epoch {epoch+1:2d}/{epochs}  loss={total_loss / len(train_loader):.4f}")

    classifier.eval()
    h_test_dev = h_test.to(device)
    y_test_dev = y_test.to(device)
    with torch.no_grad():
        preds = classifier(h_test_dev).argmax(dim=1)
        accuracy = (preds == y_test_dev).float().mean().item()

    return accuracy

In [ ]:
if RUN_SIMCLR_TRAINING:
    print("Linear evaluation of SimCLR encoder:")
    simclr_accuracy = linear_eval(
        simclr_encoder, train_ds_supervised, test_ds, device,
        epochs=20, lr=1e-2,
    )
    print(f"\nSimCLR linear eval test accuracy: {simclr_accuracy:.4f}")

#### Subpart 1.3.d: Supervised Baseline


In [ ]:
class SupervisedModel(nn.Module):
    """Encoder + linear classifier, trained end-to-end."""

    def __init__(self, num_blocks=NUM_BLOCKS, base_channels=BASE_CHANNELS, num_classes=10):
        super().__init__()
        self.encoder = SimCLREncoder(num_blocks, base_channels)
        self.fc = nn.Linear(base_channels, num_classes)

    def forward(self, x):
        h = self.encoder(x)
        return self.fc(h)


def train_supervised(model, train_dataset, test_dataset, device,
                     epochs=10, lr=1e-3, batch_size=512):
    """Standard supervised training loop."""
    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True,
        num_workers=0, pin_memory=True,
    )
    test_loader = DataLoader(
        test_dataset, batch_size=512, shuffle=False, num_workers=0,
    )

    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        if (epoch + 1) % 5 == 0:
            model.eval()
            correct = 0
            total = 0
            with torch.no_grad():
                for x, y in test_loader:
                    x, y = x.to(device), y.to(device)
                    preds = model(x).argmax(dim=1)
                    correct += (preds == y).sum().item()
                    total += y.size(0)
            print(f"  Epoch {epoch+1:2d}/{epochs}  loss={total_loss/len(train_loader):.4f}  "
                  f"test_acc={correct/total:.4f}")

    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            preds = model(x).argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    return correct / total

In [ ]:
if RUN_SIMCLR_TRAINING:
    torch.manual_seed(SEED)
    sup_model = SupervisedModel().to(device)
    print("Supervised baseline training, it may takes ~20 minutes on CPU:")
    sup_accuracy = train_supervised(
        sup_model, train_ds_supervised, test_ds, device,
        epochs=10, lr=1e-3,
    )
    print(f"\nSupervised test accuracy: {sup_accuracy:.4f}")

#### Subpart 1.3.e: Comparison


In [ ]:
if RUN_SIMCLR_TRAINING:
    print("=" * 50)
    print("Results Summary")
    print("=" * 50)
    print(f"SimCLR (linear eval):     {simclr_accuracy:.4f}")
    print(f"Supervised (end-to-end):  {sup_accuracy:.4f}")
    print(f"Gap:                      {sup_accuracy - simclr_accuracy:.4f}")
    print()
    print("SimCLR learns useful representations without any labels.")
    print("The gap to supervised training is expected — SimCLR uses only")
    print("a linear classifier on frozen features, while supervised")
    print("training jointly optimizes the encoder and classifier.")

    methods = ["SimCLR\n(linear eval)", "Supervised\n(end-to-end)"]
    accs = [simclr_accuracy, sup_accuracy]

    plt.figure(figsize=(5, 4))
    bars = plt.bar(methods, accs, color=["#2196F3", "#FF9800"], width=0.5)
    for bar, acc in zip(bars, accs):
        plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                 f"{acc:.1%}", ha="center", va="bottom", fontweight="bold")
    plt.ylabel("Test Accuracy")
    plt.title("FashionMNIST: SimCLR vs Supervised")
    plt.ylim(0, 1.0)
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

## Problem 2: GANs


### Overview and Roadmap

This part is shorter than the SimCLR section, but it still follows a clear implementation scaffold rather than a single monolithic training block.

#### What this problem asks you to do

1. Set up the MNIST training data and latent-noise dimension.
2. Define a generator and discriminator.
3. Implement the discriminator loss on real and fake batches.
4. Implement the generator loss that encourages fake samples to look real.
5. Alternate the two updates inside a training loop and visualize generated digits.

#### Reading guide

- The model-definition cells tell you what objects are being optimized.
- The loss cells isolate the two optimization objectives.
- The `train_one_epoch` cell is the key place where the alternating-update logic is implemented.


### Section 2.1: Practical GAN Implementation


#### Subpart 2.1.a: Setup and Data Loading

These cells choose a device, load MNIST, normalize images to the `[-1, 1]` range expected by the generator output, and build the dataloader used during training.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
batch_size = 128

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

#### Subpart 2.1.b: Generator and Discriminator

The generator maps latent noise vectors to flattened images, while the discriminator maps flattened images to real/fake probabilities. Keep the input and output shapes in mind while reading the implementation.


In [ ]:
# TODO: Define the generator and discriminator networks used for GAN training.
# The generator maps latent noise of shape (batch_size, latent_dim) to flattened images of shape (batch_size, 28 * 28).
# The discriminator maps flattened images of shape (batch_size, 28 * 28) to probabilities of shape (batch_size, 1).

latent_dim = 100
image_dim = 28 * 28

# sol
# YOUR CODE HERE
pass
# los



In [ ]:
lr = 2e-4

optimizer_G = optim.Adam(G.parameters(), lr=lr)
optimizer_D = optim.Adam(D.parameters(), lr=lr)

criterion = nn.BCELoss()

In [ ]:
def show_generated_images(generator, num_images=16):
    generator.eval()
    with torch.no_grad():
        z = torch.randn(num_images, latent_dim, device=device)
        fake_images = generator(z).view(-1, 28, 28).cpu()

    fig, axes = plt.subplots(4, 4, figsize=(6, 6))
    for ax, img in zip(axes.flatten(), fake_images):
        ax.imshow(img, cmap="gray")
        ax.axis("off")
    plt.show()
    generator.train()

#### Subpart 2.1.c: Loss Functions

These cells implement the two optimization objectives separately: one for the discriminator and one for the generator.


In [ ]:
def discriminator_loss(D, real_images, fake_images, criterion):
    # TODO: Implement the discriminator loss.
    # real_images: (B, 784), fake_images: (B, 784)
    # The discriminator should output a scalar probability for each image, so the targets have shape (B, 1).
    # sol
    # YOUR CODE HERE
    pass
    # los
    return loss_D



In [ ]:
def generator_loss(D, fake_images, criterion):
    # TODO: Implement the generator loss.
    # fake_images has shape (B, 784). The generator wants D(fake_images) to be close to 1 for each sample.
    # sol
    # YOUR CODE HERE
    pass
    # los

    return loss_G



#### Subpart 2.1.d: One Epoch of Alternating Updates

This is the core GAN training scaffold. The discriminator is updated on real images and detached fake images, then the generator is updated using a fresh batch of latent noise.


In [ ]:
def train_one_epoch(G, D, dataloader, optimizer_G, optimizer_D, criterion, latent_dim, device):
    G.train()
    D.train()

    total_loss_G = 0.0
    total_loss_D = 0.0

    for real_images, _ in dataloader:
        # TODO: Implement one full GAN training step.
        # 1. Flatten the real images to shape (B, 784).
        # 2. Sample latent noise z with shape (B, latent_dim).
        # 3. Update the discriminator using real images and detached fake images.
        # 4. Sample fresh latent noise and update the generator.
        # sol
        # YOUR CODE HERE
        pass
        # los

        total_loss_D += loss_D.item()
        total_loss_G += loss_G.item()

    avg_loss_D = total_loss_D / len(dataloader)
    avg_loss_G = total_loss_G / len(dataloader)

    return avg_loss_G, avg_loss_D



#### Subpart 2.1.e: Full Training Run and Visualization

The final loop calls `train_one_epoch`, prints the generator and discriminator losses, and periodically visualizes generated samples so students can connect the code to the qualitative behavior of GAN training.


In [ ]:
num_epochs = 20

for epoch in range(num_epochs):
    avg_loss_G, avg_loss_D = train_one_epoch(
        G, D, dataloader, optimizer_G, optimizer_D, criterion, latent_dim, device
    )

    print(f"Epoch {epoch+1}/{num_epochs} | Loss D: {avg_loss_D:.4f} | Loss G: {avg_loss_G:.4f}")

    if (epoch + 1) % 2 == 0:
        show_generated_images(G)

## Problem 3: Clustering


### Overview and Roadmap

This section is the most scaffold-heavy because there are several numbered subparts, several plots, and two different algorithms. The goal is for the markdown to tell you exactly which code cells correspond to which written question.

#### What this problem asks you to do

1. Implement K-means and verify its objective decreases over iterations.
2. Visualize cluster means across multiple random restarts.
3. Repeat after standardizing pixels and compare the centroids.
4. Implement HAC with max, min, and centroid linkage on the smaller dataset.
5. Compare cluster sizes and confusion matrices across methods.
6. Use those plots to support the qualitative discussion questions in the writeup.

#### Data note

The original scaffold expects `data/large_dataset.npy`, `data/small_dataset.npy`, and `data/small_dataset_labels.npy`. The cell below checks for those files and recreates them from the local raw MNIST files if needed so the instructions and the code stay consistent.


This section contains the K-means and HAC implementation and the plotting code used for the clustering problem.


### Setup: Dataset Preparation


In [ ]:
from pathlib import Path

def ensure_clustering_datasets(data_dir="data"):
    data_dir = Path(data_dir)
    large_path = data_dir / "large_dataset.npy"
    small_path = data_dir / "small_dataset.npy"
    small_labels_path = data_dir / "small_dataset_labels.npy"

    if large_path.exists() and small_path.exists() and small_labels_path.exists():
        return

    raw_dir = data_dir / "MNIST" / "raw"
    image_file = raw_dir / "train-images-idx3-ubyte"
    label_file = raw_dir / "train-labels-idx1-ubyte"

    with open(image_file, "rb") as f:
        image_bytes = f.read()
    images = np.frombuffer(image_bytes, dtype=np.uint8, offset=16).reshape(-1, 28 * 28).astype(np.float32)

    with open(label_file, "rb") as f:
        label_bytes = f.read()
    labels = np.frombuffer(label_bytes, dtype=np.uint8, offset=8).astype(np.int64)

    np.save(large_path, images[:5000])
    np.save(small_path, images[:300])
    np.save(small_labels_path, labels[:300])

ensure_clustering_datasets()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from seaborn import heatmap
from scipy.spatial import distance

### Setup: Load Clustering Datasets


In [ ]:
# Loading datasets

small_dataset = np.load("data/small_dataset.npy")
small_labels = np.load("data/small_dataset_labels.npy").astype(int)
large_dataset = np.load("data/large_dataset.npy")

### Section 3.1: K-Means


#### K-Means implementation

The `KMeans` class below corresponds to the first three clustering subproblems. Its helper methods match the standard alternating procedure: initialize means, compute distances, update assignments, and then recompute means. The later plotting cells call methods from this class directly, so this is the core implementation to understand first.


In [ ]:
class KMeans(object):
    # K is the K in KMeans
    def __init__(self, K):
        self.K = K

    # TODO: Implement the helper methods and fit loop for K-means.
    # Keep track of distances, assignments, cluster means, and the objective value at each iteration.

    # Some helper functions may go here:

    # sol
    # YOUR CODE HERE
    pass
    # los

    def fit(self, X):
        """
        Run the k-Means algorithm for exactly 10 iterations.

        :param X: a (N x 784) array since the dimension of each image is 28x28
        """
        # sol
        # YOUR CODE HERE
        pass
        # los
        
    def plot_verify_objective(self):
        """
        This should plot the objective as a function of iteration and verify that it never increases.

        This assumes that fit() has already been called.
        """
        # sol
        # YOUR CODE HERE
        pass
        # los

    def get_mean_images(self):
        """ 
        This should return the arrays for K images. Each image should represent the mean of each of the fitted clusters.
        """
        # sol
        # YOUR CODE HERE
        pass
        # los

    def get_cluster_sizes(self):
        # sol
        # YOUR CODE HERE
        pass
        # los



#### Subpart 3.1.a: Verify the K-means Objective


In [ ]:
np.random.seed(181)
KMeansClassifier = KMeans(K=10)
KMeansClassifier.fit(large_dataset)
KMeansClassifier.plot_verify_objective()

#### Helper for Subparts 3.1.b and 3.1.c


In [ ]:
def make_mean_image_plot(data, standardized = False, filename=None):
    niters = 3
    K = 10
    allmeans = np.zeros((K, niters, 784))
    for i in range(niters):
        KMeansClassifier = KMeans(K=K)
        KMeansClassifier.fit(data)
        allmeans[:,i] = KMeansClassifier.get_mean_images()
    fig = plt.figure(figsize=(10,10))
    plt.suptitle('Class mean images across random restarts' + (' (standardized data)' if standardized else ''), fontsize=16)
    for k in range(K):
        for i in range(niters):
            ax = fig.add_subplot(K, niters, 1+niters*k+i)
            plt.setp(ax.get_xticklabels(), visible=False)
            plt.setp(ax.get_yticklabels(), visible=False)
            ax.tick_params(axis='both', which='both', length=0)
            if k == 0: plt.title('Iter '+str(i))
            if i == 0: ax.set_ylabel('Class '+str(k), rotation=90)
            plt.imshow(allmeans[k,i].reshape(28,28), cmap='Greys_r')
    if filename:
        plt.savefig(f'img_output/{filename}') 
    plt.show()

#### Subpart 3.1.b: Mean Images from Raw Pixels


In [ ]:
make_mean_image_plot(large_dataset, filename='p2.2.png')

#### Subpart 3.1.c: Mean Images after Standardization


In [ ]:
# TODO: Standardize each pixel to have mean 0 and variance 1 before rerunning K-means.
# For any zero-variance pixel, divide by 1 instead to avoid division by zero.
large_dataset_std = None 

# sol
# YOUR CODE HERE
pass
# los

make_mean_image_plot(large_dataset_std, standardized = True, filename='p2.3.png')


### Section 3.2: Hierarchical Agglomerative Clustering


For HAC, the main scaffold difference is that merges happen between clusters rather than between points and centroids. The implementation below tracks the cluster assignments over time so later cells can recover the partition when there are exactly 10 clusters.


In [ ]:
class HAC(object):
    # TODO: Implement HAC with support for max, min, and centroid linkage.
    # Track the assignment history so later cells can recover the clustering when there are exactly 10 clusters.
    def __init__(self, linkage):
        self.linkage = linkage
        # sol
        # YOUR CODE HERE
        pass
        # los

    def fit(self, X):
        # sol
        # YOUR CODE HERE
        pass
        # los

    def get_k_clusters(self, k):
        # sol
        # YOUR CODE HERE
        pass
        # los

    def get_mean_images(self, n_clusters):
        # sol
        # YOUR CODE HERE
        pass
        # los

    def get_cluster_sizes(self, n_clusters):
        # sol
        # YOUR CODE HERE
        pass
        # los



#### Subpart 3.2.a: HAC Mean Images


In [ ]:
LINKAGES = [ 'max', 'min', 'centroid' ]
n_clusters = 10
cluster_sizes = []

fig = plt.figure(figsize=(10,10))
plt.suptitle("HAC mean images with max, min, and centroid linkagess")
for l_idx, l in enumerate(LINKAGES):
    # Fit HAC
    hac = HAC(l)
    hac.fit(small_dataset)
    mean_images = hac.get_mean_images(n_clusters)
    cluster_sizes.append(hac.get_cluster_sizes(n_clusters)) # used in Part 8, append here to avoid re-fitting
    # Make plot
    for m_idx in range(mean_images.shape[0]):
        m = mean_images[m_idx]
        ax = fig.add_subplot(n_clusters, len(LINKAGES), l_idx + m_idx*len(LINKAGES) + 1)
        plt.setp(ax.get_xticklabels(), visible=False)
        plt.setp(ax.get_yticklabels(), visible=False)
        ax.tick_params(axis='both', which='both', length=0)
        if m_idx == 0: plt.title(l)
        if l_idx == 0: ax.set_ylabel('Class '+str(m_idx), rotation=90)
        plt.imshow(m.reshape(28,28), cmap='Greys_r')
    print("Done:", l)
plt.savefig('img_output/p2.4.png') 
plt.show()

#### Subpart 3.2.b: Cluster-Size Plots


In [ ]:
# HAC graphs
def plot_cluster_sizes(n_clusters, cluster_sizes, filename=None):
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3)
    fig.suptitle("HAC Cluster Sizes")
    ax1.set_ylabel('number of images in cluster')
    cluster_idxs = [i for i in range(n_clusters)]

    def plot_sizes_per_linkage(ax, sizes, linkage):
        ax.bar(cluster_idxs, sizes)
        ax.set_title(f'{linkage} Linkage')
        ax.set_xlabel('cluster index')
        ax.set_ylim(0, 300)

    plot_sizes_per_linkage(ax1, cluster_sizes[0], 'Max')
    plot_sizes_per_linkage(ax2, cluster_sizes[1], 'Min')
    plot_sizes_per_linkage(ax3, cluster_sizes[2], 'Centroid')
    if filename:
        plt.savefig(f'img_output/{filename}') 
    plt.show()
plot_cluster_sizes(n_clusters, cluster_sizes, filename='p2.5a.png')

In [ ]:
# K-means graphs
def plot_cluster_sizes(n_clusters, cluster_sizes, filename=None):
    fig, (ax) = plt.subplots(1, 1)
    fig.suptitle("K-means Cluster Sizes")
    ax.set_ylabel('number of images in cluster')
    cluster_idxs = [i for i in range(n_clusters)]

    def plot_sizes_per_linkage(ax, sizes, k):
        ax.bar(cluster_idxs, sizes)
        ax.set_title(f'K = {k}')
        ax.set_xlabel('cluster index')
        ax.set_ylim(0, 1000)

    plot_sizes_per_linkage(ax, cluster_sizes, str(n_clusters))
    if filename:
        plt.savefig(f'img_output/{filename}') 
    plt.show()

plot_cluster_sizes(KMeansClassifier.K, KMeansClassifier.get_cluster_sizes(), filename='p2.5b.png')

#### Subpart 3.2.c: Confusion Matrices


In [ ]:
# TODO: Build the pairwise confusion matrices between K-means and the HAC variants.
# Each confusion matrix should count how often one method's cluster labels co-occur with another method's labels.
# sol
# YOUR CODE HERE
pass
# los


## Problem 4: PCA


### Overview and Roadmap

This section implements PCA from scratch on MNIST and keeps the same scaffold-first style as the earlier problems. The goal is to make it easy to connect each code block to the written PCA prompt in the PDF.

#### What this problem asks you to do

1. Compute PCA on the first 6000 MNIST training images.
2. Plot the top eigenvalues and the cumulative fraction of variance explained.
3. Visualize the mean image and the first 10 principal components.
4. Compare reconstruction error from the mean image and from the top 10 principal components.
5. Use those outputs to discuss the structure captured by PCA.

#### Reading guide

- The setup cells load the dataset and flatten the images.
- The next cells implement PCA and the helper statistics step by step.
- The final cells generate the plots and reconstruction-error numbers used in the writeup.


### Setup: Load the MNIST Data for PCA


In [ ]:
import torch
import torchvision
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = 'svg'

mnist_trainset = torchvision.datasets.MNIST(root='./data', train=True, download=True)  # download MNIST
N = 6000 

x = mnist_trainset.data[:N]  # select N datapoints
x = x.flatten(1)             # flatten the images
x = x.float()                # convert pixels from uint8 to float
# x = x.numpy()              # uncomment to use numpy (optional)

### Section 4.1: Compute PCA


#### Subpart 4.1.a: PCA via SVD


In [ ]:
def pca(x, n_comps=500):
    # TODO: Compute PCA using an SVD of the centered data matrix.
    # x has shape (N, D), the singular vectors in Vt have shape (D, D),
    # and the returned principal components should have shape (n_comps, D).
    # sol
    # YOUR CODE HERE
    pass
    # los



#### Subpart 4.1.b: Cumulative Fraction of Variance Explained


In [ ]:
def calc_cfvs(eigvals):
    # TODO: Convert the eigenvalues into cumulative fractions of variance explained.
    # eigvals has shape (n_comps,), and the return value should also have shape (n_comps,).
    # sol
    # YOUR CODE HERE
    pass
    # los



#### Subpart 4.1.c: Reconstruction Errors


In [ ]:
def calc_errs(x, pcomps):
    # TODO: Compute the reconstruction error from the mean image and from the top 10 principal components.
    # x has shape (N, D), and pcomps[:10] has shape (10, D).
    # sol
    # YOUR CODE HERE
    pass
    # los
    
    return err_mean, err_pcomp



### Section 4.2: Plotting and Discussion Outputs


#### Helper: Plot the Mean Image, Principal Components, and Variance Curves


In [ ]:
def plot_pic(pic, ax, title=''):
    # TODO: Reshape a flattened 784-dimensional vector back into a 28 x 28 image for display.
    # sol
    # YOUR CODE HERE
    pass
    # los

def make_plots(eigvals, cfvs, x_mean, pcomps):
    # TODO: Create the PCA figures used in the writeup.
    # 1. Plot the leading eigenvalues and cumulative fraction of variance explained.
    # 2. Plot the mean image.
    # 3. Plot the first 10 principal components.
    # sol
    # YOUR CODE HERE
    pass
    # los



#### Final PCA Run and Saved Figures


In [ ]:
# TODO: Run the PCA pipeline end to end and save the figures used in the writeup.
# sol
# YOUR CODE HERE
pass
# los

